# Modules 2–5 — Retrieval, generation and evaluation decisions

This notebook continues `01_ingestion_decisions.ipynb`. It documents **why** chunking, retrieval, answer generation and evaluation work the way they do. Each finding follows the same pattern:

- **Question** — what we were unsure about
- **Evidence** — code run against the corpus or against recorded evaluation runs
- **Decision** — what we chose, and why

**Prerequisites.** The full pipeline must have been run (corpus, chunks, index, LitQA2 benchmark), and the evaluation runs listed in the setup cell must exist under `experiments/results/`. Findings 2 and 3 load the retriever, and Finding 3 calls the local model through Ollama, so Ollama must be running. The notebook never re-runs evaluations; it reads the recorded results, each of which stores its full configuration.

**Outputs and licensing.** Outputs are stripped on commit by `nbstripout`, except for cells marked `keep_output`. Kept cells print only counts, scores, identifiers and metrics — never article text, LitQA2 question text or model answers. Cells that print such text are deliberately not kept.

**Test-split rule.** Individual results from the test split are never examined, here or anywhere else. Only test-split summaries appear in this notebook, so the test split never influences a decision.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path
from statistics import median

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
# Pipeline code uses paths relative to the repository root.
os.chdir(ROOT)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "experiments" / "results"


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def find_run(pattern: str) -> Path:
    """Resolve a run ID or pattern to exactly one results folder."""
    matches = sorted(RESULTS.glob(pattern))
    if len(matches) != 1:
        names = [m.name for m in matches]
        raise ValueError(f"{pattern!r} matched {len(matches)} runs: {names}. Pin the exact run ID.")
    return matches[0]


# The evaluation runs this notebook documents. Pin exact IDs, so later runs
# with similar names cannot silently replace them.
RUNS = {
    "tune_dense": "20261021T114958Z_tune",
    "test_dense": "20261021T141338Z_test",
    "tune_hybrid": "*_tune_hybrid",
    "test_hybrid": "20261022T085055Z_test_hybrid",
}
run_dirs = {name: find_run(pattern) for name, pattern in RUNS.items()}
summaries = {name: json.loads((d / "summary.json").read_text()) for name, d in run_dirs.items()}
configs = {name: json.loads((d / "config.json").read_text()) for name, d in run_dirs.items()}

for name, path in run_dirs.items():
    print(f"{name:<12} {path.name}")

tune_dense   20261021T114958Z_tune
test_dense   20261021T141338Z_test
tune_hybrid  20261021T144518Z_tune_hybrid
test_hybrid  20261022T085055Z_test_hybrid


## Finding 1 — Chunk size

**Question.** A chunk is the unit retrieval compares a question against, and the unit the model receives as evidence. How large should it be?

**Constraint.** The embedding model (`bge-small-en-v1.5`) reads at most 512 tokens and silently ignores the rest. Too-small chunks lack context; too-large chunks mix topics and embed as a vague average.

**Approach.** Consecutive paragraphs from the same section are merged up to ~300 words (about 400 tokens), leaving room for a prefix with the paper title and section path. Captions are separate chunks. No overlap.

**Evidence.** The resulting chunk distribution on the current corpus:

In [ ]:
chunks = read_jsonl(PROCESSED / "chunks.jsonl")
sizes = [len(c["text"].split()) for c in chunks]
per_paper = Counter(c["pmcid"] for c in chunks)
short = sum(size < 20 for size in sizes)

print(f"Papers: {len(per_paper)} | chunks: {len(chunks)}")
print(f"Words per chunk: min {min(sizes)}, median {median(sizes):.0f}, max {max(sizes)}")
print(f"Chunks under 20 words: {short} ({short / len(sizes):.1%})")
print(f"Chunks per paper: median {median(per_paper.values()):.0f}, max {max(per_paper.values())}")

Papers: 138 | chunks: 6492
Words per chunk: min 1, median 170, max 300
Chunks under 20 words: 251 (3.9%)
Chunks per paper: median 47, max 111


**Decision.** Keep ~300-word, section-aware chunks with a title and section prefix for embedding. Chunks rarely approach the token limit, and most are well below 300 words because sections end first. Very short chunks (figure labels without captions, tails of split paragraphs) are a known minor issue; chunk size and overlap remain open experiments, to be measured on the benchmark.

## Finding 2 — Retrieval scores and the off-topic gate

**Question.** Does embedding search find the right material, and can its similarity scores tell answerable questions from unanswerable ones?

**Evidence.** Four hand-written probe questions, retrieved with embedding search only, as at the time of the decision. The corpus has grown since (from 49 to 138 papers), so exact scores may differ slightly from those first observed.

In [ ]:
from evidence_rag_assistant.retrieval.retriever import Retriever

PROBES = {
    "specific (one paper)": "How does DyAb predict antibody properties with little data?",
    "broad (many papers)": "How accurate are methods for predicting antibody structures?",
    "on-topic, unanswerable": "What was the serum half-life of DyAb-designed antibodies in mice?",
    "off-topic": "What is the capital of France?",
}

dense = Retriever(mode="dense")
for label, question in PROBES.items():
    results = dense.retrieve(question, k=5)
    scores = [r.score for r in results]
    papers = Counter(r.chunk["pmcid"] for r in results)
    print(f"{label:<24} top {max(scores):.3f}  lowest {min(scores):.3f}  papers {dict(papers)}")

/home/nick/personal_dir/evidence-rag-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2696.78it/s]


specific (one paper)     top 0.867  lowest 0.851  papers {'PMC13531962': 5}
broad (many papers)      top 0.840  lowest 0.823  papers {'PMC13600136': 3, 'PMC12279266': 2}
on-topic, unanswerable   top 0.784  lowest 0.774  papers {'PMC13531962': 5}
off-topic                top 0.449  lowest 0.435  papers {'PMC11077085': 3, 'PMC9729108': 1, 'PMC9605863': 1}


**Findings.**

- **Specific questions retrieve well:** all five chunks come from the one relevant paper.
- **Broad questions lack diversity:** a few papers — often reviews — fill all five slots.
- **Off-topic questions score far lower** than on-topic ones, so a score threshold can catch them.
- **On-topic but unanswerable questions score almost as high as answerable ones.** A threshold cannot separate them; only the model, reading the passages, can.
- Scores are a ranking, not a probability: even unrelated text scores around 0.4.

**Decision.** Two layers of abstention: a provisional retrieval threshold (0.6) for off-topic questions, and an instruction for the model to reply with a fixed sentence when the passages do not answer the question.

## Finding 3 — Grounded answers: citations and abstention

**Question.** Does the full system — retrieval, gate, model — behave as designed on the same probes?

**Evidence.** The probes answered by the current system (hybrid retrieval, Qwen3 8B via Ollama, temperature 0). Only statuses, scores and citation numbers are kept; the answer texts are printed by the next, unkept cell.

In [ ]:
from evidence_rag_assistant.generation.answer import MIN_RETRIEVAL_SCORE, answer_question

system = Retriever()  # current default: hybrid retrieval
answers = {label: answer_question(question, system) for label, question in PROBES.items()}

print(f"Retrieval gate threshold: {MIN_RETRIEVAL_SCORE}\n")
for label, answer in answers.items():
    print(
        f"{label:<24} {answer.status:<24} top {answer.top_score:.3f}  "
        f"cited {answer.cited}  invalid {answer.invalid_citations}"
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4812.76it/s]


Retrieval gate threshold: 0.6

specific (one paper)     answered                 top 0.867  cited [1, 2, 4, 5]  invalid []
broad (many papers)      answered                 top 0.840  cited [3]  invalid []
on-topic, unanswerable   abstained_by_model       top 0.784  cited []  invalid []
off-topic                abstained_low_retrieval  top 0.449  cited []  invalid []


In [ ]:
# Not kept: prints model answers, which paraphrase article text.
for label, answer in answers.items():
    print(f"--- {label}\n{answer.text}\n")

**Expected pattern.** The specific and broad questions are `answered` with valid citations; the off-topic question is stopped by the gate (`abstained_low_retrieval`) without calling the model; the on-topic unanswerable question passes the gate and is declined by the model (`abstained_by_model`).

**Decisions.**

- Sources are numbered `[1]`–`[5]` rather than labelled with chunk IDs: short numbers are copied reliably by a model, and are mapped back to papers afterwards.
- Temperature 0, so the same question gives the same answer.
- Citations are validated: numbers outside the provided sources are flagged, and an answer without any valid citation is labelled `ungrounded`.

## Finding 4 — Choosing a benchmark

**Question.** Judging answers by eye on a few probes cannot show whether a change helps. Which benchmark gives many questions with known answers and sources, validated by people, without writing them by hand?

**Options considered.**

| Dataset | Assessment |
|---|---|
| **LitQA2** (FutureHouse, in LAB-Bench) | Expert-written multiple-choice questions on recent papers; answers in the full text; source DOI per question; an "insufficient information" option |
| PubMedQA | Expert-labelled, but abstracts only, with yes/no/maybe answers |
| BioASQ | Expert-curated, but requires registration and restricts redistribution |

**Decision.** LitQA2. A 2026 study had already used its Europe PMC full-text subset — the same source and format as this pipeline.

**How it is used.** LitQA2 source papers and the 49 antibody papers share one index. LitQA2 questions are evaluated against it, so the antibody papers act as distractors, and the tuned system is exactly the system later used for antibody questions. No accuracy is claimed for antibody questions themselves.

## Finding 5 — From LitQA2 to corpus

**Question.** How many LitQA2 questions can be evaluated, given that every source paper must pass the same licence verification as the rest of the corpus?

In [ ]:
candidates = read_jsonl(RAW / "litqa2_candidates.jsonl")
found = [c for c in candidates if c["title"] is not None]
with_pmcid = [c for c in candidates if c["pmcid"]]
full_text = [c for c in candidates if c["in_epmc"] == "Y"]
eligible = [c for c in with_pmcid if c["in_epmc"] == "Y" and c["licence_ok"]]

print("LitQA2 source DOIs → Europe PMC")
print(f"  distinct source DOIs:  {len(candidates)}")
print(f"  found in Europe PMC:   {len(found)}")
print(f"  with a PMCID:          {len(with_pmcid)}")
print(f"  full text in EPMC:     {len(full_text)}")
print(f"  eligible to fetch:     {len(eligible)}")

manifest = read_jsonl(RAW / "fulltext_manifest.jsonl")
print("\nFetch status by corpus:")
for (source, status), n in sorted(Counter((e.get("source"), e["status"]) for e in manifest).items()):
    print(f"  {str(source):<12} {status:<20} {n}")

benchmark = read_jsonl(PROCESSED / "litqa2_benchmark.jsonl")
print(f"\nBenchmark questions: {len(benchmark)} | split: {dict(Counter(q['split'] for q in benchmark))}")

LitQA2 source DOIs → Europe PMC
  distinct source DOIs:  191
  found in Europe PMC:   188
  with a PMCID:          129
  full text in EPMC:     130
  eligible to fetch:     99

Fetch status by corpus:
  Europe PMC   cached               49
  Europe PMC   licence_unverified   1
  LitQA2       fetch_error          9
  LitQA2       licence_unverified   1
  LitQA2       stored               89

Benchmark questions: 89 | split: {'tune': 38, 'test': 51}


**Findings.**

- The largest loss is papers without a PMCID — mostly preprints, which Europe PMC lists without full text.
- Few papers are lost at download, because selecting on PMCID and Europe-PMC-hosted text screens out PDF-only articles beforehand.
- Licence verification behaves as for the antibody papers: fail-closed, with no disagreement between index and article licences.

**Decisions.**

- Keep a question only if **all** its source papers are in the corpus.
- Shuffle answer options per question with a fixed seed, and grade by the chosen option letter — the dataset contains a misspelled correct answer, so matching answer text would be wrong.
- Split questions permanently into tune (~40%) and test (~60%) by hashing their IDs. Decisions are made on tune only; test is for reporting.

## Finding 6 — Evaluation design

**Question.** How can one benchmark test answering, abstaining, and whether answers come from retrieval or from the model's memory?

**Decision: three modes per question.**

| Mode | The model receives | Correct behaviour |
|---|---|---|
| `rag` | The top five retrieved chunks | Pick the correct option |
| `source_removed` | The top five chunks, excluding the source paper | Pick "insufficient information" |
| `closed_book` | No sources | Mostly "insufficient"; correct answers here come from memory |

`source_removed` turns every question into an on-topic unanswerable one without writing new questions. `closed_book` checks for contamination: LitQA2 was public before the model was released.

**Metrics.** Accuracy (correct / all), precision (correct / answered), source hit (source paper retrieved), key-passage hit (the sentence containing the answer retrieved, by ≥80% word overlap), and diversity (distinct papers among the top five). Retrieval and answer metrics are separated so that a failure can be attributed to finding the evidence or to using it.

**Traceability.** Every run records split, settings, models, the corpus hash and the Git commit next to its results.

## Finding 7 — An operational problem: running out of memory

**Question.** The first full evaluation crashed partway, and generation had slowed to about 1.6 tokens per second. Why?

**Evidence (system-level, not reproducible in this notebook).**

- The model ran 100% on the GPU with VRAM to spare — the GPU was not the problem.
- WSL had about 1 GB of RAM free and most of its swap in use.
- The kernel log showed the model runner killed for running out of memory.
- The runner's log showed a prompt cache in RAM growing by 200–350 MB per question.

**Decision.** Disable llama.cpp's prompt cache through the Ollama service configuration (`LLAMA_ARG_CACHE_RAM=0`); it only helps with repeated prompts, which an evaluation never has. Generation speed returned to about 62 tokens per second, and evaluations complete. The evaluation code was also made resilient: model calls are retried, and a question that still fails is recorded as an error rather than crashing the run.

## Finding 8 — The dense baseline

**Question.** How well does the simplest complete system — embedding search, five chunks, 300-word chunks — perform?

In [ ]:
METRICS = {
    "rag": ["n", "correct", "wrong", "abstained", "accuracy", "precision",
            "source_hit_rate", "key_passage_hit_rate", "key_passage_checked",
            "mean_distinct_papers"],
    "source_removed": ["n", "abstained", "wrong", "expected_behaviour_rate"],
    "closed_book": ["n", "correct", "wrong", "abstained", "accuracy", "precision"],
}


def show(names: list[str]) -> None:
    for mode, keys in METRICS.items():
        print(f"\n{mode}")
        print(f"  {'':<24}" + "".join(f"{name:>14}" for name in names))
        for key in keys:
            values = [summaries[name].get(mode, {}).get(key) for name in names]
            cells = "".join(
                f"{v:>14.3f}" if isinstance(v, float) else f"{str(v):>14}" for v in values
            )
            print(f"  {key:<24}{cells}")


for name in ["tune_dense", "test_dense"]:
    config = configs[name]
    print(
        f"{name}: commit {config['git']['commit'][:7]}, "
        f"uncommitted changes: {config['git']['uncommitted_changes']}, "
        f"retrieval: {config.get('retrieval', 'dense (not recorded)')}, k: {config['k']}"
    )
show(["tune_dense", "test_dense"])

tune_dense: commit 03e5907, uncommitted changes: True, retrieval: dense (not recorded), k: 5
test_dense: commit 36c7ec3, uncommitted changes: True, retrieval: dense (not recorded), k: 5

rag
                              tune_dense    test_dense
  n                                   38            51
  correct                             32            40
  wrong                                1             0
  abstained                            5            11
  accuracy                         0.842         0.784
  precision                        0.970         1.000
  source_hit_rate                      1             1
  key_passage_hit_rate             0.789         0.780
  key_passage_checked               None            50
  mean_distinct_papers             1.184         1.118

source_removed
                              tune_dense    test_dense
  n                                   38            51
  abstained                           38            51
  wrong                

**Findings.**

- High accuracy and very high precision: when the system answers, it is almost always right.
- With the source paper removed, the system declined every question — no invented answers.
- Closed-book accuracy is far below RAG accuracy, so the answers come from retrieval, not from the model's memory.
- The source paper was always retrieved, and the top five chunks almost always came from it alone.
- The tune run was started with uncommitted changes (the retry logic); later runs start from committed code.

**What this result is not.** LitQA2's authors reported 66% accuracy for their system, which had to find the right paper in the open literature. Here, each question's source paper is guaranteed to be in a 138-paper corpus. The results describe a closed-corpus setting and are not comparable with open-literature results. Because finding the right paper is already solved here, the metric with room to improve is the **key-passage hit**: finding the right passage within the paper.

## Finding 9 — What the dense failures teach (tune split)

**Question.** Why did the dense system miss the questions it missed? Only tune-split results are examined.

In [ ]:
tune_dense = [r for r in read_jsonl(run_dirs["tune_dense"] / "results.jsonl") if r["mode"] == "rag"]
failures = [r for r in tune_dense if r["outcome"] != "correct"]

print(f"{'id':<10} {'outcome':<10} {'key passage':<12} top score")
for r in failures:
    print(f"{r['id'][:8]:<10} {r['outcome']:<10} {str(r['key_passage_hit']):<12} {r['top_score']:.3f}")

retrieval_misses = sum(not r["key_passage_hit"] for r in failures)
print(f"\nFailures where the key passage was not retrieved: {retrieval_misses} / {len(failures)}")

id         outcome    key passage  top score
39129e1c   abstained  False        0.821
cb710074   abstained  True         0.837
322454df   abstained  False        0.862
f0b6cea0   wrong      False        0.836
398ebac1   abstained  True         0.835
c6e11fac   abstained  False        0.811

Failures where the key passage was not retrieved: 4 / 6


In [ ]:
# Not kept: prints the end of each failing model answer, which paraphrases article text.
for r in failures:
    print(f"=== {r['id'][:8]}  {r['outcome']}\n{r['raw_answer'][-400:]}\n")

**Findings.**

- Most failures are **retrieval failures**: the right paper was retrieved, but not the passage containing the answer. In most of these, the model saw only related information and correctly declined.
- The one wrong answer was an **extrapolation**: the model noted that the asked-for organism group was not mentioned, then answered with a value for a broader group anyway. A single case is too few to justify changing the prompt.
- The failing questions hinge on **rare, exact terms** — an organism group, a mutation code, a specific temperature. Embedding search captures overall meaning and blurs such terms; keyword search weights rare terms heavily.

**Decision.** Add keyword search (BM25) and combine it with embedding search (hybrid retrieval), judged first by the key-passage hit rate.

## Finding 10 — Hybrid retrieval

**Question.** Does combining keyword search with embedding search find the answer passage more often — without making the system less careful?

**Approach.** BM25 ranks chunks by shared words, weighting rare words heavily. The two rankings are combined with **Reciprocal Rank Fusion**, which uses positions only (each chunk scores `1 / (60 + rank)` per ranking), because cosine and BM25 scores are on incompatible scales. Each chunk keeps its embedding score, and the off-topic gate uses the best embedding score among the returned chunks.

**Evidence 1 — tune split, summary.**

In [ ]:
show(["tune_dense", "tune_hybrid"])


rag
                              tune_dense   tune_hybrid
  n                                   38            38
  correct                             32            33
  wrong                                1             1
  abstained                            5             4
  accuracy                         0.842         0.868
  precision                        0.970         0.971
  source_hit_rate                      1             1
  key_passage_hit_rate             0.789         0.895
  key_passage_checked               None            38
  mean_distinct_papers             1.184         1.158

source_removed
                              tune_dense   tune_hybrid
  n                                   38            38
  abstained                           38            38
  wrong                                0             0
  expected_behaviour_rate          1.000         1.000

closed_book
                              tune_dense   tune_hybrid
  n                            

**Evidence 2 — tune split, question by question.** Totals can hide changes that cancel out, so the same questions are compared under both settings. Only questions whose outcome or key-passage hit changed are listed.

In [ ]:
def rag_results(name: str) -> dict[str, dict]:
    rows = read_jsonl(run_dirs[name] / "results.jsonl")
    return {r["id"]: r for r in rows if r["mode"] == "rag"}


before, after = rag_results("tune_dense"), rag_results("tune_hybrid")
print(f"{'id':<10} {'dense':<24} {'hybrid':<24}")
for qid in sorted(before):
    b, a = before[qid], after[qid]
    if (b["outcome"], b["key_passage_hit"]) != (a["outcome"], a["key_passage_hit"]):
        print(
            f"{qid[:8]:<10} {b['outcome']:<10} kp={str(b['key_passage_hit']):<10}"
            f" {a['outcome']:<10} kp={a['key_passage_hit']}"
        )

id         dense                    hybrid                  
39129e1c   abstained  kp=False      correct    kp=True
398ebac1   abstained  kp=True       wrong      kp=True
58f69c8d   correct    kp=False      correct    kp=True
7975ddb0   correct    kp=False      correct    kp=True
8b665114   correct    kp=True       abstained  kp=False
c6e11fac   abstained  kp=False      correct    kp=True
d7833c0f   correct    kp=False      correct    kp=True
f0b6cea0   wrong      kp=False      abstained  kp=False


**Tune findings.**

- Key-passage hits: five gained, one lost.
- Two retrieval failures became correct answers; the only wrong answer became a safe abstention.
- Two regressions: one question lost its key passage (pushed out of the top five by keyword-promoted chunks) and abstained; one question with the passage present went from abstaining to a wrong answer.
- Net: better retrieval, one more correct answer, the same number of wrong answers. The decision was to adopt hybrid retrieval and confirm on the test split.

**Evidence 3 — test split, summaries only.** The test split was run once per configuration; its individual results are not examined.

In [ ]:
show(["test_dense", "test_hybrid"])


rag
                              test_dense   test_hybrid
  n                                   51            51
  correct                             40            44
  wrong                                0             1
  abstained                           11             6
  accuracy                         0.784         0.863
  precision                        1.000         0.978
  source_hit_rate                      1             1
  key_passage_hit_rate             0.780         0.920
  key_passage_checked                 50            50
  mean_distinct_papers             1.118         1.098

source_removed
                              test_dense   test_hybrid
  n                                   51            51
  abstained                           51            51
  wrong                                0             0
  expected_behaviour_rate          1.000         1.000

closed_book
                              test_dense   test_hybrid
  n                            

In [ ]:
def totals(names: list[str]) -> dict[str, float]:
    rag = [summaries[name]["rag"] for name in names]
    # Runs before the missing-key-passage fix did not record key_passage_checked;
    # their hit rate was computed over all n questions (every tune question has one).
    checked_per_run = [s.get("key_passage_checked", s["n"]) for s in rag]
    checked = sum(checked_per_run)
    hits = sum(
        round(s["key_passage_hit_rate"] * c) for s, c in zip(rag, checked_per_run, strict=True)
    )
    n = sum(s["n"] for s in rag)
    removed_wrong = sum(summaries[name]["source_removed"]["wrong"] for name in names)
    return {
        "questions": n,
        "correct": sum(s["correct"] for s in rag),
        "wrong": sum(s["wrong"] for s in rag),
        "accuracy": sum(s["correct"] for s in rag) / n,
        "key passage hit": hits / checked,
        "invented answers (source removed)": removed_wrong,
    }

KeyError: 'key_passage_checked'

**Findings.**

- On the held-out test split, hybrid retrieval raised accuracy and found the answer passage far more often — a larger gain than on tune, on questions that never influenced the decision.
- Wrong answers rose by one across all questions; precision stays very high.
- With the source paper removed, no answer was invented, in either configuration.
- Closed-book results are identical between runs: with temperature 0 the system is deterministic, so differences come from the retrieval change, not randomness.

**Decision.** Hybrid retrieval becomes the default. Embedding-only search remains available (`--retrieval dense`), so the baseline stays reproducible.

**Open question.** One tune question lost its answer passage because keyword-promoted chunks displaced it from the top five. Whether retrieving more chunks recovers such cases — or distracts the model — is the next experiment.

## Summary of decisions

| Area | Decision | Evidence |
|---|---|---|
| Chunking | ~300-word section-aware chunks, title and section prefix | Embedding model's 512-token limit; chunk distribution (Finding 1) |
| Abstention | Retrieval gate for off-topic questions + model abstention for unanswerable ones | Score gap for off-topic, no gap for on-topic unanswerable (Findings 2–3) |
| Citations | Numbered sources, validated; uncited answers flagged | Finding 3 |
| Benchmark | LitQA2, Europe PMC full-text subset, shared index with antibody papers | Finding 4 |
| Benchmark build | All source papers present; shuffled options graded by letter; permanent tune/test split | Finding 5 |
| Evaluation | `rag`, `source_removed` and `closed_book` modes; retrieval and answer metrics separated | Finding 6 |
| Operations | Prompt cache disabled; retries and recorded errors | Finding 7 |
| Baseline | Dense retrieval: closed-corpus results, not comparable with open-literature results | Finding 8 |
| Retrieval | Hybrid BM25 + embeddings with Reciprocal Rank Fusion, now the default | Findings 9–10 |